# ResNet-50 with Attention    
## Binary DR Detection (당뇨병성 망막병증 이진 분류 모델)
###dataset_1_ver1

---

**데이터 구조:**
- Train: 1,920 images
- Val: 640 images  
- Test: 640 images
- Label: 0 (No DR), 1 (DR)


In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 환경설정

import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingWarmRestarts

# Torchvision
import torchvision.transforms as transforms
from torchvision import models

# 이미지 처리
from PIL import Image
import cv2

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 평가 지표
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)



In [ ]:
# 환경설정_2

# 재현성을 위한 시드 설정
SEED = 42

def set_seed(seed=42):
    """재현성을 위한 랜덤 시드 고정"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# GPU 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")



🔧 Device: cuda
   GPU: NVIDIA L4
   Memory: 23.80 GB


In [ ]:
# ==============================================================================
# 1. 경로 및 하이퍼파라미터 설정
# ==============================================================================

# 데이터 경로
BASE_DIR = '/content/drive/MyDrive/ColabNotebooks/RFMiD_A/EDA_2/dataset_1'

# CSV 경로
TRAIN_CSV = os.path.join(BASE_DIR, 'manifest_train.csv')
VAL_CSV = os.path.join(BASE_DIR, 'manifest_val.csv')
TEST_CSV = os.path.join(BASE_DIR, 'manifest_test.csv')

# 이미지 디렉토리 (전처리된 512x512 이미지)
TRAIN_IMG_DIR = os.path.join(BASE_DIR, 'train')
VAL_IMG_DIR = os.path.join(BASE_DIR, 'val')
TEST_IMG_DIR = os.path.join(BASE_DIR, 'test')

# 체크포인트 저장 경로
CHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints_resnet50_attention')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# 하이퍼파라미터
CONFIG = {
    # 모델
    'model_name': 'resnet50_attention',
    'num_classes': 1,  # Binary classification
    'pretrained': True,

    # 데이터
    'img_size': 512,
    'batch_size': 16,
    'num_workers': 4,

    # 학습
    'epochs': 50,
    'learning_rate': 1e-4,  # ResNet-50은 EfficientNet보다 높은 LR 가능
    'weight_decay': 1e-4,
    'patience': 10,

    # 손실 함수
    'use_focal_loss': False,  # True로 바꾸면 Focal Loss 사용
    'focal_alpha': 0.25,
    'focal_gamma': 2.0,

    # Gradient clipping
    'max_grad_norm': 1.0,

    # 기타
    'save_best_only': True
}

print("\n" + "="*80)
print("📋 CONFIGURATION")
print("="*80)
for key, value in CONFIG.items():
    print(f"  {key:20s}: {value}")
print("="*80)



📋 CONFIGURATION
  model_name          : resnet50_attention
  num_classes         : 1
  pretrained          : True
  img_size            : 512
  batch_size          : 16
  num_workers         : 4
  epochs              : 50
  learning_rate       : 0.0001
  weight_decay        : 0.0001
  patience            : 10
  use_focal_loss      : False
  focal_alpha         : 0.25
  focal_gamma         : 2.0
  max_grad_norm       : 1.0
  save_best_only      : True


In [ ]:
# ==============================================================================
# 2. Dataset 클래스
# ==============================================================================

class FundusDataset(Dataset):
    """
    안저 영상 Dataset 클래스

    특징:
    - Auto column detection (ID, DR 등)
    - Image file format detection (.png, .jpg)
    - Robust error handling
    """

    def __init__(self, csv_path, img_dir, transform=None, is_test=False):
        """
        Args:
            csv_path (str): CSV 파일 경로
            img_dir (str): 이미지 디렉토리 경로
            transform (callable): 이미지 변환 함수
            is_test (bool): 테스트 모드 (라벨 없음)
        """
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

        # CSV 로드
        self.df = pd.read_csv(csv_path)

        # 컬럼명 자동 감지
        self._detect_columns()

        # 통계 출력
        if not is_test:
            print(f"\n📊 Dataset: {os.path.basename(csv_path)}")
            print(f"   Total: {len(self.df)} samples")
            print(f"   DR positive: {self.df[self.label_col].sum()} ({self.df[self.label_col].sum()/len(self.df)*100:.1f}%)")
            print(f"   DR negative: {(1-self.df[self.label_col]).sum()} ({(1-self.df[self.label_col]).sum()/len(self.df)*100:.1f}%)")

    def _detect_columns(self):
        """CSV 컬럼명 자동 감지"""
        columns = self.df.columns.tolist()

        # Image ID 컬럼 찾기
        img_col_candidates = ['img_id', 'ID', 'id', 'image_id', 'Image_ID']
        self.img_col = None
        for col in img_col_candidates:
            if col in columns:
                self.img_col = col
                break

        if self.img_col is None:
            raise ValueError(f"Image ID column not found! Available columns: {columns}")

        # Label 컬럼 찾기 (test set은 제외)
        if not self.is_test:
            label_col_candidates = ['label', 'DR', 'target', 'class', 'Label']
            self.label_col = None
            for col in label_col_candidates:
                if col in columns:
                    self.label_col = col
                    break

            if self.label_col is None:
                raise ValueError(f"Label column not found! Available columns: {columns}")

        print(f"   ✓ Image column: '{self.img_col}'")
        if not self.is_test:
            print(f"   ✓ Label column: '{self.label_col}'")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # 이미지 ID
        img_id = str(self.df.iloc[idx][self.img_col])

        # 이미지 경로 (확장자 자동 감지)
        img_path = None
        for ext in ['.png', '.jpg', '.jpeg', '.PNG', '.JPG']:
            candidate_path = os.path.join(self.img_dir, f"{img_id}{ext}")
            if os.path.exists(candidate_path):
                img_path = candidate_path
                break

        if img_path is None:
            raise FileNotFoundError(f"Image not found: {img_id}")

        # 이미지 로드
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"⚠️ Error loading {img_path}: {e}")
            # 에러 시 검은색 이미지 반환
            image = Image.new('RGB', (CONFIG['img_size'], CONFIG['img_size']), (0, 0, 0))

        # Transform 적용
        if self.transform:
            image = self.transform(image)

        # 라벨
        if self.is_test:
            return image, img_id
        else:
            label = float(self.df.iloc[idx][self.label_col])
            return image, torch.tensor(label, dtype=torch.float32)



In [ ]:
# ==============================================================================
# 3. Data Augmentation
# ==============================================================================

def get_transforms(mode='train', img_size=512):
    """
    의료 영상 특화 Data Augmentation

    Args:
        mode (str): 'train' or 'val'/'test'
        img_size (int): 이미지 크기

    Returns:
        transforms.Compose: 변환 파이프라인
    """
    if mode == 'train':
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),  # 안저 영상은 회전에 강건
            transforms.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.1,
                hue=0.05
            ),
            # 추가: Random Gaussian Blur (노이즈 시뮬레이션)
            # transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],  # ImageNet 표준
                std=[0.229, 0.224, 0.225]
            )
        ])
    else:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

In [ ]:
# ==============================================================================
# 4. Attention Modules
# ==============================================================================

class ChannelAttention(nn.Module):
    """
    Channel Attention Module (CAM)

    채널 간 중요도를 학습하여 병변과 관련된 채널을 강조
    Reference: CBAM (Convolutional Block Attention Module)
    """

    def __init__(self, in_channels, reduction_ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction_ratio, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction_ratio, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # 평균 풀링 경로
        avg_out = self.fc(self.avg_pool(x))
        # 최대 풀링 경로
        max_out = self.fc(self.max_pool(x))
        # 결합
        out = avg_out + max_out
        return self.sigmoid(out) * x


class SpatialAttention(nn.Module):
    """
    Spatial Attention Module (SAM)

    공간적 위치의 중요도를 학습하여 병변 위치를 강조
    """

    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # 채널 축으로 평균/최대 계산
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        # 채널 방향으로 concat
        out = torch.cat([avg_out, max_out], dim=1)
        out = self.conv(out)
        return self.sigmoid(out) * x


class CBAM(nn.Module):
    """
    Convolutional Block Attention Module

    Channel Attention → Spatial Attention 순차 적용
    """

    def __init__(self, in_channels, reduction_ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.ca = ChannelAttention(in_channels, reduction_ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.ca(x)
        x = self.sa(x)
        return x

In [ ]:
# ==============================================================================
# 5. ResNet-50 with Attention Model
# ==============================================================================

class ResNet50Attention(nn.Module):
    """
    ResNet-50 with CBAM Attention

    구조:
    1. ResNet-50 백본 (ImageNet pretrained)
    2. CBAM attention 추가
    3. Global Average Pooling
    4. Fully Connected Layer

    특징:
    - Deep network (50 layers) → 복잡한 패턴 학습 가능
    - Attention mechanism → 병변 위치 강조
    - Transfer learning → 적은 데이터로도 높은 성능
    """

    def __init__(self, num_classes=1, pretrained=True):
        super(ResNet50Attention, self).__init__()

        # ResNet-50 백본 로드
        self.backbone = models.resnet50(pretrained=pretrained)

        # Attention 추가 (layer4 출력에 적용)
        # layer4의 출력 채널: 2048
        self.attention = CBAM(in_channels=2048, reduction_ratio=16)

        # 마지막 FC layer 제거 (직접 구성)
        self.backbone.fc = nn.Identity()

        # Global Average Pooling
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(2048, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # ResNet-50 특징 추출
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)

        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        x = self.backbone.layer4(x)  # [B, 2048, H, W]

        # Attention 적용
        x = self.attention(x)  # [B, 2048, H, W]

        # Global Average Pooling
        x = self.global_pool(x)  # [B, 2048, 1, 1]
        x = torch.flatten(x, 1)   # [B, 2048]

        # Classification
        x = self.classifier(x)    # [B, num_classes]

        return x


In [ ]:
# ==============================================================================
# 6. Focal Loss (선택적)
# ==============================================================================

class FocalLoss(nn.Module):
    """
    Focal Loss for Binary Classification

    클래스 불균형 문제 해결:
    - 쉬운 샘플의 loss 가중치 감소
    - 어려운 샘플에 집중

    Formula:
        FL(p_t) = -α_t * (1 - p_t)^γ * log(p_t)
    """

    def __init__(self, alpha=0.25, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1 - pt) ** self.gamma * BCE_loss
        return F_loss.mean()

In [ ]:
# ==============================================================================
# 7. 학습 및 평가 함수
# ==============================================================================

def train_one_epoch(model, dataloader, criterion, optimizer, device, config):
    """1 에폭 학습"""
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    pbar = tqdm(dataloader, desc='Training', leave=False)

    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)  # [B] -> [B, 1]

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), config['max_grad_norm'])

        optimizer.step()

        # 통계
        running_loss += loss.item() * images.size(0)
        preds = torch.sigmoid(outputs).detach().cpu().numpy()
        all_preds.extend(preds.flatten())
        all_labels.extend(labels.cpu().numpy().flatten())

        # 진행률 업데이트
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    # 에폭 평균
    epoch_loss = running_loss / len(dataloader.dataset)

    # 메트릭 계산
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    pred_binary = (all_preds > 0.5).astype(int)

    accuracy = accuracy_score(all_labels, pred_binary)
    auc = roc_auc_score(all_labels, all_preds)

    return epoch_loss, accuracy, auc


def validate(model, dataloader, criterion, device):
    """검증"""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Validation', leave=False)

        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = torch.sigmoid(outputs).cpu().numpy()
            all_preds.extend(preds.flatten())
            all_labels.extend(labels.cpu().numpy().flatten())

    # 평균
    epoch_loss = running_loss / len(dataloader.dataset)

    # 메트릭
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    pred_binary = (all_preds > 0.5).astype(int)

    accuracy = accuracy_score(all_labels, pred_binary)
    precision = precision_score(all_labels, pred_binary, zero_division=0)
    recall = recall_score(all_labels, pred_binary, zero_division=0)
    f1 = f1_score(all_labels, pred_binary, zero_division=0)
    auc = roc_auc_score(all_labels, all_preds)

    return epoch_loss, accuracy, precision, recall, f1, auc





In [ ]:
# ==============================================================================
# 8. 메인 학습 루프
# ==============================================================================

def train_model(model, train_loader, val_loader, config, device):
    """전체 학습 프로세스"""

    print("\n" + "="*80)
    print("🚀 TRAINING START")
    print("="*80)

    # 손실 함수
    if config['use_focal_loss']:
        criterion = FocalLoss(alpha=config['focal_alpha'], gamma=config['focal_gamma'])
        print("📌 Loss: Focal Loss")
    else:
        # Weighted BCE Loss (클래스 불균형 대응)
        # pos_weight 계산
        train_labels = []
        for _, labels in train_loader:
            train_labels.extend(labels.numpy())
        pos_count = sum(train_labels)
        neg_count = len(train_labels) - pos_count
        pos_weight = torch.tensor([neg_count / pos_count]).to(device)
        pos_weight = torch.clamp(pos_weight, max=5.0)  # 최대 5배로 제한

        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        print(f"📌 Loss: Weighted BCE (pos_weight={pos_weight.item():.2f})")

    # Optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )

    # Scheduler
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode='max',  # AUC 기준
        factor=0.5,
        patience=5
    )

    # 학습 히스토리
    history = {
        'train_loss': [],
        'train_acc': [],
        'train_auc': [],
        'val_loss': [],
        'val_acc': [],
        'val_precision': [],
        'val_recall': [],
        'val_f1': [],
        'val_auc': [],
        'lr': []
    }

    best_auc = 0.0
    patience_counter = 0

    # 학습 시작
    for epoch in range(config['epochs']):
        print(f"\n{'='*80}")
        print(f"Epoch {epoch+1}/{config['epochs']}")
        print(f"{'='*80}")

        # 학습
        train_loss, train_acc, train_auc = train_one_epoch(
            model, train_loader, criterion, optimizer, device, config
        )

        # 검증
        val_loss, val_acc, val_prec, val_rec, val_f1, val_auc = validate(
            model, val_loader, criterion, device
        )

        # Learning rate 업데이트
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_auc)

        # 히스토리 저장
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_auc'].append(train_auc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_precision'].append(val_prec)
        history['val_recall'].append(val_rec)
        history['val_f1'].append(val_f1)
        history['val_auc'].append(val_auc)
        history['lr'].append(current_lr)

        # 결과 출력
        print(f"\n📊 Results:")
        print(f"   Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | AUC: {train_auc:.4f}")
        print(f"   Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | AUC: {val_auc:.4f}")
        print(f"   Val   Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f}")
        print(f"   LR: {current_lr:.6f}")

        # 체크포인트 저장
        if val_auc > best_auc:
            best_auc = val_auc
            patience_counter = 0

            checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_auc': val_auc,
                'val_acc': val_acc,
                'config': config
            }

            save_path = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
            torch.save(checkpoint, save_path)
            print(f"\n✅ Best model saved! (AUC: {best_auc:.4f})")
        else:
            patience_counter += 1
            print(f"\n⏳ No improvement. Patience: {patience_counter}/{config['patience']}")

        # Early stopping
        if patience_counter >= config['patience']:
            print(f"\n🛑 Early stopping triggered after {epoch+1} epochs")
            break

    print("\n" + "="*80)
    print("✅ TRAINING COMPLETE")
    print("="*80)
    print(f"Best Validation AUC: {best_auc:.4f}")

    return history, best_auc



In [ ]:
# ==============================================================================
# 9. 메인 실행
# ==============================================================================

def main():
    """메인 실행 함수"""

    print("\n" + "="*80)
    print("ResNet-50 with Attention - Binary DR Detection")
    print("="*80)

    # ========================================
    # 1. 데이터 로드
    # ========================================
    print("\n📂 Loading datasets...")

    # Transforms
    train_transform = get_transforms(mode='train', img_size=CONFIG['img_size'])
    val_transform = get_transforms(mode='val', img_size=CONFIG['img_size'])

    # Datasets
    train_dataset = FundusDataset(TRAIN_CSV, TRAIN_IMG_DIR, transform=train_transform)
    val_dataset = FundusDataset(VAL_CSV, VAL_IMG_DIR, transform=val_transform)

    # DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=True,
        num_workers=CONFIG['num_workers'],
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True
    )

    print(f"\n✅ Data loaded successfully!")
    print(f"   Train batches: {len(train_loader)}")
    print(f"   Val batches: {len(val_loader)}")

    # ========================================
    # 2. 모델 생성
    # ========================================
    print("\n🏗️ Building model...")

    model = ResNet50Attention(
        num_classes=CONFIG['num_classes'],
        pretrained=CONFIG['pretrained']
    )
    model = model.to(device)

    # 모델 정보
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\n✅ Model created!")
    print(f"   Total parameters: {total_params:,}")
    print(f"   Trainable parameters: {trainable_params:,}")

    # ========================================
    # 3. 학습
    # ========================================
    history, best_auc = train_model(model, train_loader, val_loader, CONFIG, device)

    # ========================================
    # 4. 학습 곡선 시각화
    # ========================================
    print("\n📈 Plotting training curves...")

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # Loss
    axes[0, 0].plot(history['train_loss'], label='Train', linewidth=2)
    axes[0, 0].plot(history['val_loss'], label='Validation', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Loss Curve')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Accuracy
    axes[0, 1].plot(history['train_acc'], label='Train', linewidth=2)
    axes[0, 1].plot(history['val_acc'], label='Validation', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].set_title('Accuracy Curve')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # AUC
    axes[1, 0].plot(history['train_auc'], label='Train', linewidth=2)
    axes[1, 0].plot(history['val_auc'], label='Validation', linewidth=2)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('AUC')
    axes[1, 0].set_title('AUC Curve')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Learning Rate
    axes[1, 1].plot(history['lr'], linewidth=2, color='red')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].set_title('Learning Rate Schedule')
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(CHECKPOINT_DIR, 'training_curves.png'), dpi=150)
    print(f"   ✅ Saved to: {os.path.join(CHECKPOINT_DIR, 'training_curves.png')}")

    # ========================================
    # 5. 최종 결과 저장
    # ========================================
    results = {
        'model': CONFIG['model_name'],
        'best_val_auc': best_auc,
        'total_epochs': len(history['train_loss']),
        'config': CONFIG
    }

    results_df = pd.DataFrame([results])
    results_df.to_csv(os.path.join(CHECKPOINT_DIR, 'results.csv'), index=False)

    print("\n" + "="*80)
    print("🎉 ALL DONE!")
    print("="*80)
    print(f"Checkpoint directory: {CHECKPOINT_DIR}")
    print(f"Best model: best_model.pth (AUC: {best_auc:.4f})")
    print("="*80)


In [ ]:
# ==============================================================================
# 실행
# ==============================================================================

if __name__ == '__main__':
    main()



"\nif __name__ == '__main__':\n    main()\n\n    "

In [ ]:



from google.colab import drive
drive.mount('/content/drive')

import os
import torch
import torch.nn as nn
import torch.nn.functional as F


# ==============================================================================
# 1. 경로 설정
# ==============================================================================

# ⚠️ 본인의 경로로 수정!
BASE_DIR = '/content/drive/MyDrive/ColabNotebooks/RFMiD_A/EDA_2/dataset_1'
MODEL_DIR = '/content/drive/MyDrive/ColabNotebooks/RFMiD_A/model/ResNet-50_Attention/ver1/checkpoints_resnet50_attention'

TEST_CSV = os.path.join(BASE_DIR, 'manifest_test.csv')
TEST_IMG_DIR = os.path.join(BASE_DIR, 'test')
MODEL_PATH = os.path.join(MODEL_DIR, 'best_model.pth')

# ==============================================================================
# 2. Dataset & Model
# ==============================================================================

class FundusDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = str(row['img_id'])
        label = float(row['label'])

        img_path = os.path.join(self.img_dir, f"{img_id}.png")

        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (512, 512))

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)


class AttentionModule(nn.Module):
    def __init__(self, in_channels):
        super(AttentionModule, self).__init__()
        self.attention = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 8, kernel_size=1),
            nn.BatchNorm2d(in_channels // 8),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 8, in_channels, kernel_size=1),
            nn.BatchNorm2d(in_channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        att = self.attention(x)
        return x * att


class ResNet50Attention(nn.Module):
    """
    ResNet-50 with CBAM Attention

    구조:
    1. ResNet-50 백본 (ImageNet pretrained)
    2. CBAM attention 추가
    3. Global Average Pooling
    4. Fully Connected Layer

    특징:
    - Deep network (50 layers) → 복잡한 패턴 학습 가능
    - Attention mechanism → 병변 위치 강조
    - Transfer learning → 적은 데이터로도 높은 성능
    """

    def __init__(self, num_classes=1, pretrained=True):
        super(ResNet50Attention, self).__init__()

        # ResNet-50 백본 로드
        self.backbone = models.resnet50(pretrained=pretrained)

        # Attention 추가 (layer4 출력에 적용)
        # layer4의 출력 채널: 2048
        self.attention = CBAM(in_channels=2048, reduction_ratio=16)

        # 마지막 FC layer 제거 (직접 구성)
        self.backbone.fc = nn.Identity()

        # Global Average Pooling
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(2048, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # ResNet-50 특징 추출
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)

        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        x = self.backbone.layer4(x)  # [B, 2048, H, W]

        # Attention 적용
        x = self.attention(x)  # [B, 2048, H, W]

        # Global Average Pooling
        x = self.global_pool(x)  # [B, 2048, 1, 1]
        x = torch.flatten(x, 1)   # [B, 2048]

        # Classification
        x = self.classifier(x)    # [B, num_classes]

        return x


# ==============================================================================
# 3. Test 평가 함수
# ==============================================================================

def evaluate_test_set(model, test_loader, device):
    """Test set 평가"""
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc='Testing'):
            images = images.to(device)
            outputs = model(images).squeeze(1)
            probs = torch.sigmoid(outputs)

            all_preds.extend(probs.cpu().numpy())
            all_labels.extend(labels.numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    return all_preds, all_labels


def compute_metrics(labels, probs, threshold=0.5):
    """메트릭 계산"""
    preds = (probs >= threshold).astype(int)

    accuracy = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    auc = roc_auc_score(labels, probs)

    cm = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc,
        'specificity': specificity,
        'confusion_matrix': cm,
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp
    }


def plot_results(metrics, checkpoint_info, output_dir):
    """결과 시각화"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Confusion Matrix
    cm = metrics['confusion_matrix']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['No DR', 'DR'], yticklabels=['No DR', 'DR'])
    axes[0].set_title('Test Set Confusion Matrix', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('True Label')
    axes[0].set_xlabel('Predicted Label')

    # Metrics Bar Chart
    metric_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'Specificity', 'AUC']
    metric_values = [
        metrics['accuracy'], metrics['precision'], metrics['recall'],
        metrics['f1'], metrics['specificity'], metrics['auc']
    ]

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    bars = axes[1].bar(metric_names, metric_values, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    axes[1].axhline(y=0.85, color='red', linestyle='--', linewidth=2, label='Target (85%)')
    axes[1].set_ylim([0, 1.05])
    axes[1].set_title('Test Set Performance Metrics', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Score')
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)

    # 값 표시
    for bar in bars:
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.3f}',
                    ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'test_results.png'), dpi=150, bbox_inches='tight')
    print(f"\n✅ Results plot saved!")
    plt.show()

# ==============================================================================
# 4. 메인 실행
# ==============================================================================

def main():
    print("\n" + "="*80)
    print("📊 ResNet-50 with Attention Test Set Evaluation")
    print("="*80)

    # 1. Test dataset 준비
    test_transform = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    test_dataset = FundusDataset(TEST_CSV, TEST_IMG_DIR, test_transform)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

    print(f"\n📂 Test Dataset:")
    print(f"   Total: {len(test_dataset)} samples")
    test_df = pd.read_csv(TEST_CSV)
    print(f"   No DR: {(test_df['label']==0).sum()} ({(test_df['label']==0).sum()/len(test_df)*100:.1f}%)")
    print(f"   DR:    {(test_df['label']==1).sum()} ({(test_df['label']==1).sum()/len(test_df)*100:.1f}%)")

    # 2. Best model 로드
    class ResNet50Attention_BinaryDR(nn.Module):
        def __init__(self, pretrained=True):
        super().__init__()


    print(f"\n📂 Loading Best Model...")
    model = ResNet50Attention_BinaryDR(pretrained=False)
    checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()

    print(f"✅ Model loaded successfully!")
    print(f"\n📊 Checkpoint Info:")
    print(f"   Epoch: {checkpoint['epoch']}")
    print(f"   Val Recall: {checkpoint['val_recall']:.4f}")
    print(f"   Val AUC: {checkpoint['val_auc']:.4f}")
    print(f"   Val Accuracy: {checkpoint['val_accuracy']:.4f}")
    print(f"   Val Precision: {checkpoint['val_precision']:.4f}")

    # 3. Test 평가
    print(f"\n🚀 Evaluating on Test Set...")
    all_probs, all_labels = evaluate_test_set(model, test_loader, device)

    # 4. 메트릭 계산
    metrics = compute_metrics(all_labels, all_probs, threshold=0.5)

    # 5. 결과 출력
    print("\n" + "="*80)
    print("📊 TEST SET PERFORMANCE")
    print("="*80)
    print(f"Accuracy:    {metrics['accuracy']:.4f}")
    print(f"Precision:   {metrics['precision']:.4f}")
    print(f"Recall:      {metrics['recall']:.4f}", end="")
    if metrics['recall'] >= 0.85:
        print(" ✅ (Target achieved!)")
    else:
        print(f" ⚠️ (Target: ≥0.85)")
    print(f"F1 Score:    {metrics['f1']:.4f}")
    print(f"Specificity: {metrics['specificity']:.4f}")
    print(f"AUC:         {metrics['auc']:.4f}")

    print(f"\n📊 Confusion Matrix:")
    print(metrics['confusion_matrix'])
    print(f"\nTN={metrics['tn']}, FP={metrics['fp']}, FN={metrics['fn']}, TP={metrics['tp']}")
    print(f"False Negative Rate: {metrics['fn']/(metrics['fn']+metrics['tp'])*100:.2f}% ({metrics['fn']} patients missed)")

    # 6. Validation vs Test 비교
    print("\n" + "="*80)
    print("📊 VALIDATION vs TEST COMPARISON")
    print("="*80)
    print(f"{'Metric':<15} {'Validation':<15} {'Test':<15} {'Difference':<15}")
    print("-"*80)
    print(f"{'Recall':<15} {checkpoint['val_recall']:<15.4f} {metrics['recall']:<15.4f} {metrics['recall']-checkpoint['val_recall']:+.4f}")
    print(f"{'AUC':<15} {checkpoint['val_auc']:<15.4f} {metrics['auc']:<15.4f} {metrics['auc']-checkpoint['val_auc']:+.4f}")
    print(f"{'Accuracy':<15} {checkpoint['val_accuracy']:<15.4f} {metrics['accuracy']:<15.4f} {metrics['accuracy']-checkpoint['val_accuracy']:+.4f}")
    print(f"{'Precision':<15} {checkpoint['val_precision']:<15.4f} {metrics['precision']:<15.4f} {metrics['precision']-checkpoint['val_precision']:+.4f}")

    # 7. 평가
    recall_gap = abs(metrics['recall'] - checkpoint['val_recall'])
    auc_gap = abs(metrics['auc'] - checkpoint['val_auc'])

    print("\n" + "="*80)
    print("✅ FINAL EVALUATION")
    print("="*80)

    if metrics['recall'] >= 0.85:
        print("✅ Recall ≥ 85%: FDA/WHO criteria met!")
    else:
        print(f"⚠️ Recall < 85%: {0.85 - metrics['recall']:.2%} below target")

    if recall_gap < 0.05:
        print("✅ Val-Test Recall gap < 5%: Good generalization!")
    else:
        print(f"⚠️ Val-Test Recall gap ≥ 5%: Check for overfitting")

    if auc_gap < 0.02:
        print("✅ Val-Test AUC gap < 2%: Excellent generalization!")
    else:
        print(f"⚠️ Val-Test AUC gap ≥ 2%: Some performance drop")

    # 8. 시각화
    print(f"\n📈 Plotting results...")
    checkpoint_info = {
        'epoch': checkpoint['epoch'],
        'val_recall': checkpoint['val_recall'],
        'val_auc': checkpoint['val_auc']
    }
    plot_results(metrics, checkpoint_info, MODEL_DIR)

    # 9. 결과 저장
    results_df = pd.DataFrame({
        'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'Specificity', 'AUC',
                   'TN', 'FP', 'FN', 'TP'],
        'Value': [metrics['accuracy'], metrics['precision'], metrics['recall'],
                  metrics['f1'], metrics['specificity'], metrics['auc'],
                  metrics['tn'], metrics['fp'], metrics['fn'], metrics['tp']]
    })

    results_path = os.path.join(MODEL_DIR, 'test_results.csv')
    results_df.to_csv(results_path, index=False)
    print(f"✅ Results saved: {results_path}")

    print("\n" + "="*80)
    print("🎉 TEST EVALUATION COMPLETE!")
    print("="*80)

if __name__ == '__main__':
    main()


IndentationError: expected an indented block after function definition on line 254 (ipython-input-2854099803.py, line 255)

In [3]:
# ==============================================================================
# 📊 TEST SET EVALUATION - RN50_ver1 (완전 독립 실행)
# ==============================================================================

# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

print("\n" + "="*80)
print("📊 ResNet-50 with Attention ver1 Test Set Evaluation")
print("="*80)

# ------------------------------------------------------------------------------
# 1. 경로 설정
# ------------------------------------------------------------------------------

BASE_DIR = '/content/drive/MyDrive/ColabNotebooks/RFMiD_A/EDA_2/dataset_1'
CHECKPOINT_DIR = '/content/drive/MyDrive/ColabNotebooks/RFMiD_A/model/ResNet-50_Attention/ver1/checkpoints_resnet50_attention'

TEST_CSV = os.path.join(BASE_DIR, 'manifest_test.csv')
TEST_IMG_DIR = os.path.join(BASE_DIR, 'test')
MODEL_PATH = os.path.join(CHECKPOINT_DIR, 'best_model.pth')

IMG_SIZE = 512
BATCH_SIZE = 16

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ------------------------------------------------------------------------------
# 2. Dataset 클래스 정의
# ------------------------------------------------------------------------------

class FundusDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform

        # 컬럼명 확인
        if 'img_id' in self.df.columns:
            self.id_col = 'img_id'
        elif 'ID' in self.df.columns:
            self.id_col = 'ID'
        else:
            self.id_col = self.df.columns[0]

        if 'label' in self.df.columns:
            self.label_col = 'label'
        elif 'DR' in self.df.columns:
            self.label_col = 'DR'
        else:
            self.label_col = self.df.columns[1]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = str(row[self.id_col])  # ⭐ 문자열 변환

        # 파일 확장자 확인
        if not img_id.endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(self.img_dir, f"{img_id}.png")
            if not os.path.exists(img_path):
                img_path = os.path.join(self.img_dir, f"{img_id}.jpg")
        else:
            img_path = os.path.join(self.img_dir, img_id)

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label = row[self.label_col]
        return image, torch.tensor(label, dtype=torch.float32)

# ------------------------------------------------------------------------------
# 3. Transform 정의
# ------------------------------------------------------------------------------

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
print("✅ Transform created")

# ------------------------------------------------------------------------------
# 4. CBAM Attention 모듈 정의
# ------------------------------------------------------------------------------

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction_ratio, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction_ratio, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out) * x


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        out = self.conv(out)
        return self.sigmoid(out) * x


class CBAM(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.ca = ChannelAttention(in_channels, reduction_ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.ca(x)
        x = self.sa(x)
        return x

# ------------------------------------------------------------------------------
# 5. ResNet50Attention 모델 정의
# ------------------------------------------------------------------------------

class ResNet50Attention(nn.Module):
    def __init__(self, num_classes=1, pretrained=True):
        super(ResNet50Attention, self).__init__()
        self.backbone = models.resnet50(pretrained=pretrained)
        self.attention = CBAM(in_channels=2048, reduction_ratio=16)
        self.backbone.fc = nn.Identity()
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(2048, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)
        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        x = self.backbone.layer4(x)
        x = self.attention(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model = ResNet50Attention(num_classes=1, pretrained=False)
model = model.to(device)

# ------------------------------------------------------------------------------
# 6. Test Dataset 로드
# ------------------------------------------------------------------------------

print("\n📂 Test Dataset:")
test_df = pd.read_csv(TEST_CSV)
print(f"   Total: {len(test_df)} samples")
print(f"   No DR: {len(test_df[test_df['label']==0])} ({len(test_df[test_df['label']==0])/len(test_df)*100:.1f}%)")
print(f"   DR:    {len(test_df[test_df['label']==1])} ({len(test_df[test_df['label']==1])/len(test_df)*100:.1f}%)")

test_dataset = FundusDataset(TEST_CSV, TEST_IMG_DIR, test_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)
print(f"   Test batches: {len(test_loader)}")

# ------------------------------------------------------------------------------
# 7. Best Model 로드
# ------------------------------------------------------------------------------

print("\n📂 Loading Best Model...")
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("✅ Model loaded successfully!")
print(f"\n📊 Checkpoint Info:")
print(f"   Epoch: {checkpoint['epoch']}")
print(f"   Keys: {list(checkpoint.keys())}")

val_auc = checkpoint.get('best_auc') or checkpoint.get('val_auc')
if val_auc: print(f"   Val AUC: {val_auc:.4f}")

# ------------------------------------------------------------------------------
# 8. Test Set 평가
# ------------------------------------------------------------------------------

print("\n🚀 Evaluating on Test Set...")

all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Testing'):
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images).squeeze()
        probs = torch.sigmoid(outputs)
        preds = (probs >= 0.5).float()
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# ------------------------------------------------------------------------------
# 9. 성능 지표 계산 및 출력
# ------------------------------------------------------------------------------

test_acc = accuracy_score(all_labels, all_preds)
test_prec = precision_score(all_labels, all_preds, zero_division=0)
test_recall = recall_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds)
test_auc = roc_auc_score(all_labels, all_probs)
cm = confusion_matrix(all_labels, all_preds)

tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)

print("\n" + "="*80)
print("📊 TEST SET PERFORMANCE")
print("="*80)
print(f"Accuracy:    {test_acc:.4f}")
print(f"Precision:   {test_prec:.4f}")
print(f"Recall:      {test_recall:.4f} {'✅' if test_recall >= 0.85 else '⚠️'}")
print(f"F1 Score:    {test_f1:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"AUC:         {test_auc:.4f}")

print(f"\n📊 Confusion Matrix:")
print(f"[[{tn:3d} {fp:3d}]")
print(f" [{fn:3d} {tp:3d}]]")
print(f"\nTN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"False Negative Rate: {fn/(fn+tp)*100:.2f}% ({fn} patients missed)")

print("\n" + "="*80)
print("🎉 TEST EVALUATION COMPLETE!")
print("="*80)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

📊 ResNet-50 with Attention ver1 Test Set Evaluation
Device: cuda
✅ Transform created


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)



📂 Test Dataset:
   Total: 640 samples
   No DR: 516 (80.6%)
   DR:    124 (19.4%)
   Test batches: 40

📂 Loading Best Model...
✅ Model loaded successfully!

📊 Checkpoint Info:
   Epoch: 6
   Keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'val_auc', 'val_acc', 'config']
   Val AUC: 0.9726

🚀 Evaluating on Test Set...


Testing: 100%|██████████| 40/40 [02:31<00:00,  3.78s/it]


📊 TEST SET PERFORMANCE
Accuracy:    0.9297
Precision:   0.7970
Recall:      0.8548 ✅
F1 Score:    0.8249
Specificity: 0.9477
AUC:         0.9710

📊 Confusion Matrix:
[[489  27]
 [ 18 106]]

TN=489, FP=27, FN=18, TP=106
False Negative Rate: 14.52% (18 patients missed)

🎉 TEST EVALUATION COMPLETE!
